# 0050 v2-B3 High Risk Sample Review - CLEAN FIXED
Upload `0050_v2B3_final_stratified_review_outputs.xlsx` when prompted.


In [3]:
# -*- coding: utf-8 -*-
"""
0050 v2-B3 High Risk Sample Review - CLEAN FIXED VERSION

This script is NOT v2-C and NOT a new main strategy version.
It reviews high-risk samples of the v2-B3 final signal candidate.

Recommended input:
    0050_v2B3_final_stratified_review_outputs.xlsx

Best sheet inside input:
    Parsed_B3_Events

Output:
    0050_v2B3_high_risk_sample_review_outputs_FIXED.xlsx

Important fixes in this version:
1. Lunch tail tag is now exact-match based.
   It will NOT mistakenly tag session names like '盤中主段_1000_1245' as lunch tail.
2. The code will report missing feature columns clearly.
3. Scenario tests using missing feature columns will be skipped instead of producing misleading results.
"""

import pandas as pd
import numpy as np
import re
from IPython.display import display

try:
    from google.colab import files
    IS_COLAB = True
except Exception:
    IS_COLAB = False


# ============================================================
# 0. Parameters
# ============================================================

TICK_SIZE = 0.05
FIXED_COST_001 = 0.01
FIXED_COST_002 = 0.02
SLIP_025_TICK_ROUNDTRIP = 0.25 * TICK_SIZE * 2
SLIP_05_TICK_ROUNDTRIP = 0.50 * TICK_SIZE * 2
OUTPUT_FILE = "0050_v2B3_high_risk_sample_review_outputs_FIXED.xlsx"


# ============================================================
# 1. Helper functions
# ============================================================

def clean_number(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)
    s = str(x).strip()
    s = s.replace(",", "")
    s = s.replace("TWD", "")
    s = s.replace("%", "")
    s = s.replace("−", "-")
    s = re.sub(r"[^\d\.\-\+eE]", "", s)
    if s in ["", "-", "+", "."]:
        return np.nan
    try:
        return float(s)
    except Exception:
        return np.nan


def find_col(df, candidates):
    cols = list(df.columns)
    lower_map = {str(c).lower(): c for c in cols}
    for cand in candidates:
        if cand in cols:
            return cand
    for cand in candidates:
        cand_low = cand.lower()
        if cand_low in lower_map:
            return lower_map[cand_low]
    for cand in candidates:
        cand_low = cand.lower()
        for c in cols:
            if cand_low in str(c).lower():
                return c
    return None


def normalize_time_hhmm(x):
    if pd.isna(x):
        return np.nan
    if hasattr(x, "strftime"):
        return x.strftime("%H:%M")
    s = str(x).strip()
    if re.fullmatch(r"\d{3,4}", s):
        s = s.zfill(4)
        return s[:2] + ":" + s[2:]
    m = re.search(r"(\d{1,2}):(\d{2})", s)
    if m:
        hh = int(m.group(1))
        mm = int(m.group(2))
        return f"{hh:02d}:{mm:02d}"
    return np.nan


def time_to_minutes(hhmm):
    if pd.isna(hhmm):
        return np.nan
    try:
        hh, mm = str(hhmm).split(":")[:2]
        return int(hh) * 60 + int(mm)
    except Exception:
        return np.nan


def assign_session(mins):
    if pd.isna(mins):
        return "Unknown"
    if mins < 9 * 60 + 5:
        return "pre_0905"
    elif mins < 9 * 60 + 45:
        return "early_main_0905_0945"
    elif mins < 10 * 60:
        return "early_mid_0945_1000"
    elif mins < 12 * 60 + 45:
        return "midday_main_1000_1245"
    elif mins <= 13 * 60:
        return "lunch_tail_1245_1300"
    else:
        return "after_1300"


def is_lunch_tail_session(session_text):
    """Exact lunch-tail check. Do NOT match '1000_1245'."""
    s = str(session_text).strip()
    return (
        s == "午盤尾段_1245_1300"
        or s == "lunch_tail_1245_1300"
        or s.endswith("_1245_1300")
    )


def calc_profit_factor(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna()
    gross_profit = pnl[pnl > 0].sum()
    gross_loss = pnl[pnl < 0].sum()
    if gross_loss < 0:
        return gross_profit / abs(gross_loss)
    if gross_profit > 0 and gross_loss == 0:
        return np.inf
    return np.nan


def calc_max_drawdown(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").fillna(0)
    cum = pnl.cumsum()
    dd = cum.cummax() - cum
    return dd.max()


def perf_summary(data, pnl_col="pnl_no_cost", group_cols=None):
    def _calc(x):
        pnl = pd.to_numeric(x[pnl_col], errors="coerce").dropna()
        trades = len(pnl)
        if trades == 0:
            return pd.Series({
                "trades": 0, "wins": 0, "losses": 0, "flats": 0,
                "win_rate": np.nan, "loss_rate": np.nan,
                "net_pnl": 0.0, "avg_pnl": np.nan, "median_pnl": np.nan,
                "gross_profit": 0.0, "gross_loss": 0.0,
                "profit_factor": np.nan,
                "max_single_profit": np.nan, "max_single_loss": np.nan,
                "estimated_max_drawdown": np.nan,
            })
        wins = int((pnl > 0).sum())
        losses = int((pnl < 0).sum())
        flats = int((pnl == 0).sum())
        gp = pnl[pnl > 0].sum()
        gl = pnl[pnl < 0].sum()
        return pd.Series({
            "trades": trades,
            "wins": wins,
            "losses": losses,
            "flats": flats,
            "win_rate": wins / trades,
            "loss_rate": losses / trades,
            "net_pnl": pnl.sum(),
            "avg_pnl": pnl.mean(),
            "median_pnl": pnl.median(),
            "gross_profit": gp,
            "gross_loss": gl,
            "profit_factor": calc_profit_factor(pnl),
            "max_single_profit": pnl.max(),
            "max_single_loss": pnl.min(),
            "estimated_max_drawdown": calc_max_drawdown(pnl),
        })
    if group_cols is None:
        return _calc(data).to_frame().T
    if isinstance(group_cols, str):
        group_cols = [group_cols]
    return data.groupby(group_cols, dropna=False).apply(_calc).reset_index()


def show_table(df, title=None, sort_by=None, ascending=True, n=None):
    out = df.copy()
    if sort_by is not None and sort_by in out.columns:
        out = out.sort_values(sort_by, ascending=ascending)
    float_cols = out.select_dtypes(include=[float]).columns
    out[float_cols] = out[float_cols].round(6)
    if title:
        print("\n" + "=" * 90)
        print(title)
        print("=" * 90)
    display(out.head(n) if n else out)
    return out


# ============================================================
# 2. Load input
# ============================================================

if IS_COLAB:
    print("Please upload: 0050_v2B3_final_stratified_review_outputs.xlsx")
    uploaded = files.upload()
    input_file = list(uploaded.keys())[0]
else:
    input_file = "0050_v2B3_final_stratified_review_outputs.xlsx"

print("Loaded:", input_file)

if input_file.lower().endswith(".xlsx"):
    xls = pd.ExcelFile(input_file)
    print("Sheets:", xls.sheet_names)
    sheet_name = None
    for s in ["Parsed_B3_Events", "Parsed_B3", "B3_Events", "Parsed_Events_With_Tags"]:
        if s in xls.sheet_names:
            sheet_name = s
            break
    if sheet_name is None:
        raise ValueError("Cannot find Parsed_B3_Events sheet.")
    df = pd.read_excel(input_file, sheet_name=sheet_name)
    print("Using sheet:", sheet_name)
elif input_file.lower().endswith(".csv"):
    df = pd.read_csv(input_file)
else:
    raise ValueError("Input must be .xlsx or .csv")

print("Raw shape:", df.shape)
print("Raw columns:", list(df.columns))


# ============================================================
# 3. Standardize columns
# ============================================================

pnl_col = find_col(df, ["pnl_no_cost", "pnl_twd", "net_pnl", "pnl", "淨損益 TWD", "淨損益"])
if pnl_col is None:
    raise ValueError("Cannot find PnL column.")
df["pnl_no_cost"] = df[pnl_col].apply(clean_number)

entry_dt_col = find_col(df, ["entry_dt", "entry_datetime", "entry_time", "entry_timestamp"])
if entry_dt_col:
    df["entry_dt"] = pd.to_datetime(df[entry_dt_col], errors="coerce")
else:
    date_col = find_col(df, ["entry_date", "date"])
    hhmm_col = find_col(df, ["entry_time_hhmm", "entry_hhmm", "time_hhmm"])
    if date_col is not None and hhmm_col is not None:
        df["entry_dt"] = pd.to_datetime(df[date_col].astype(str) + " " + df[hhmm_col].astype(str), errors="coerce")
    else:
        df["entry_dt"] = pd.NaT

exit_dt_col = find_col(df, ["exit_dt", "exit_datetime", "exit_time", "exit_timestamp"])
if exit_dt_col:
    df["exit_dt"] = pd.to_datetime(df[exit_dt_col], errors="coerce")
else:
    df["exit_dt"] = pd.NaT

if "entry_date" not in df.columns:
    df["entry_date"] = df["entry_dt"].dt.date
if "entry_year" not in df.columns:
    df["entry_year"] = df["entry_dt"].dt.year
df["entry_year"] = pd.to_numeric(df["entry_year"], errors="coerce")
if "entry_month" not in df.columns:
    df["entry_month"] = df["entry_dt"].dt.to_period("M").astype(str)
if "entry_weekday" not in df.columns:
    df["entry_weekday"] = df["entry_dt"].dt.day_name()

hhmm_col = find_col(df, ["entry_time_hhmm", "entry_hhmm", "time_hhmm"])
if hhmm_col:
    df["entry_time_hhmm"] = df[hhmm_col].apply(normalize_time_hhmm)
else:
    df["entry_time_hhmm"] = df["entry_dt"].apply(normalize_time_hhmm)
df["entry_minutes"] = df["entry_time_hhmm"].apply(time_to_minutes)

if "data_split" not in df.columns:
    def assign_split(y):
        if pd.isna(y):
            return "Unknown"
        y = int(y)
        if y <= 2024:
            return "Train_2022_2024"
        if y == 2025:
            return "Validation_2025"
        if y >= 2026:
            return "Recent_Check_2026"
        return "Unknown"
    df["data_split"] = df["entry_year"].apply(assign_split)

session_col = find_col(df, ["session_block", "session"])
if session_col:
    df["session_block"] = df[session_col].astype(str)
else:
    df["session_block"] = df["entry_minutes"].apply(assign_session)

exit_col = find_col(df, ["exit_reason", "exit_type", "reason"])
if exit_col:
    df["exit_reason"] = df[exit_col].astype(str)
else:
    df["exit_reason"] = "Unknown"

sig_col = find_col(df, ["signal_count_today", "trade_count_today", "daily_signal_count"])
if sig_col:
    df["signal_count_today"] = pd.to_numeric(df[sig_col], errors="coerce")
else:
    df = df.sort_values(["entry_date", "entry_dt"])
    df["signal_count_today"] = df.groupby("entry_date").cumcount() + 1

hold_col = find_col(df, ["holding_minutes", "hold_minutes"])
if hold_col:
    df["holding_minutes"] = pd.to_numeric(df[hold_col], errors="coerce")
else:
    df["holding_minutes"] = (df["exit_dt"] - df["entry_dt"]).dt.total_seconds() / 60

rd_col = find_col(df, ["entry_relative_deviation", "relative_deviation_at_entry", "rd_at_entry", "rd"])
tw50_col = find_col(df, ["tw50_return_5m", "tw50_5m", "tw50_ret_5m"])
rv_col = find_col(df, ["realized_vol_10bar", "rv10", "realized_vol"])
slope_col = find_col(df, ["deviation_slope_1bar", "slope_1bar"])
for col in [rd_col, tw50_col, rv_col, slope_col]:
    if col:
        df[col] = pd.to_numeric(df[col], errors="coerce")

if "trade_id" not in df.columns:
    trade_col = find_col(df, ["trade_id", "Trade number", "交易編號"])
    df["trade_id"] = df[trade_col] if trade_col else np.arange(1, len(df) + 1)

df = df.sort_values(["entry_dt", "trade_id"], na_position="last").reset_index(drop=True)


# ============================================================
# 4. Add costs
# ============================================================

df["pnl_cost_001"] = df["pnl_no_cost"] - FIXED_COST_001
df["pnl_cost_002"] = df["pnl_no_cost"] - FIXED_COST_002
df["pnl_slip_025tick"] = df["pnl_no_cost"] - SLIP_025_TICK_ROUNDTRIP
df["pnl_slip_05tick"] = df["pnl_no_cost"] - SLIP_05_TICK_ROUNDTRIP


# ============================================================
# 5. Risk tags
# ============================================================

rv80 = df[rv_col].quantile(0.8) if rv_col else np.nan


def make_tags(row):
    tags = []
    pnl0 = row.get("pnl_no_cost", np.nan)
    pnl001 = row.get("pnl_cost_001", np.nan)
    pnl002 = row.get("pnl_cost_002", np.nan)
    pnl025 = row.get("pnl_slip_025tick", np.nan)

    if pd.notna(pnl0) and pnl0 < 0:
        tags.append("Loss_Trade")
    if pd.notna(pnl0) and pd.notna(pnl001) and pnl0 > 0 and pnl001 <= 0:
        tags.append("Cost_Fragile_001")
    if pd.notna(pnl0) and pd.notna(pnl002) and pnl0 > 0 and pnl002 <= 0:
        tags.append("Cost_Fragile_002")
    if pd.notna(pnl0) and pd.notna(pnl025) and pnl0 > 0 and pnl025 <= 0:
        tags.append("Slip_Fragile_025tick")
    if "EXIT_TIME" in str(row.get("exit_reason", "")):
        tags.append("EXIT_TIME_No_Reversion")
    if pd.notna(row.get("signal_count_today")) and row.get("signal_count_today") >= 2:
        tags.append("Repeated_Signal")
    if is_lunch_tail_session(row.get("session_block", "")):
        tags.append("Lunch_Tail_1245_1300")
    if rv_col and pd.notna(rv80) and pd.notna(row.get(rv_col)) and row.get(rv_col) >= rv80:
        tags.append("High_RV10_Top20pct")
    if str(row.get("data_split", "")).startswith("Train") or (
        pd.notna(row.get("entry_year")) and int(row.get("entry_year")) <= 2024
    ):
        tags.append("Train_2022_2024")
    if str(row.get("entry_time_hhmm", "")) == "09:45":
        tags.append("Boundary_Exact_0945")
    if not tags:
        tags.append("Normal_or_Low_Risk")
    return "|".join(tags)


df["risk_tags"] = df.apply(make_tags, axis=1)
for tag in [
    "Loss_Trade",
    "Cost_Fragile_001",
    "Cost_Fragile_002",
    "Slip_Fragile_025tick",
    "EXIT_TIME_No_Reversion",
    "Repeated_Signal",
    "Lunch_Tail_1245_1300",
    "High_RV10_Top20pct",
    "Train_2022_2024",
]:
    df["is_" + tag] = df["risk_tags"].str.contains(tag, regex=False, na=False)


# ============================================================
# 6. Summaries
# ============================================================

overall_no_cost = perf_summary(df, "pnl_no_cost")
overall_cost001 = perf_summary(df, "pnl_cost_001")
overall_cost002 = perf_summary(df, "pnl_cost_002")
overall_slip025 = perf_summary(df, "pnl_slip_025tick")

by_session = perf_summary(df, "pnl_no_cost", "session_block")
by_session_cost002 = perf_summary(df, "pnl_cost_002", "session_block")
by_exit = perf_summary(df, "pnl_no_cost", "exit_reason")
by_exit_cost002 = perf_summary(df, "pnl_cost_002", "exit_reason")
by_split = perf_summary(df, "pnl_no_cost", "data_split")
by_split_cost002 = perf_summary(df, "pnl_cost_002", "data_split")
by_signal_count = perf_summary(df, "pnl_no_cost", "signal_count_today")
by_signal_count_cost002 = perf_summary(df, "pnl_cost_002", "signal_count_today")

show_table(overall_no_cost, "Overall - No Cost")
show_table(overall_cost002, "Overall - Cost 0.02")
show_table(by_session_cost002, "By Session - Cost 0.02", sort_by="profit_factor")


# ============================================================
# 7. Tag summaries and interactions
# ============================================================

tagged = df.copy()
tagged["risk_tag"] = tagged["risk_tags"].str.split("|")
tagged = tagged.explode("risk_tag")

def tag_summary(pnl_col):
    return perf_summary(tagged, pnl_col, "risk_tag").sort_values("net_pnl")

tag_summary_no_cost = tag_summary("pnl_no_cost")
tag_summary_cost001 = tag_summary("pnl_cost_001")
tag_summary_cost002 = tag_summary("pnl_cost_002")
tag_summary_slip025 = tag_summary("pnl_slip_025tick")

tag_summary_main = tag_summary_no_cost[[
    "risk_tag", "trades", "net_pnl", "avg_pnl", "gross_loss", "profit_factor", "max_single_loss"
]].rename(columns={
    "trades": "tag_count",
    "net_pnl": "net_pnl_no_cost",
    "avg_pnl": "avg_pnl_no_cost",
    "gross_loss": "gross_loss_no_cost",
    "profit_factor": "pf_no_cost",
    "max_single_loss": "max_loss_no_cost",
})

tag_summary_main = (
    tag_summary_main
    .merge(tag_summary_cost001[["risk_tag", "net_pnl", "profit_factor"]].rename(columns={"net_pnl": "net_pnl_cost001", "profit_factor": "pf_cost001"}), on="risk_tag", how="left")
    .merge(tag_summary_cost002[["risk_tag", "net_pnl", "profit_factor"]].rename(columns={"net_pnl": "net_pnl_cost002", "profit_factor": "pf_cost002"}), on="risk_tag", how="left")
    .merge(tag_summary_slip025[["risk_tag", "net_pnl", "profit_factor"]].rename(columns={"net_pnl": "net_pnl_slip025", "profit_factor": "pf_slip025"}), on="risk_tag", how="left")
    .sort_values("net_pnl_cost002")
)

tag_x_split = perf_summary(tagged, "pnl_cost_002", ["risk_tag", "data_split"]).sort_values("net_pnl")
tag_x_session = perf_summary(tagged, "pnl_cost_002", ["risk_tag", "session_block"]).sort_values("net_pnl")
tag_x_exit = perf_summary(tagged, "pnl_cost_002", ["risk_tag", "exit_reason"]).sort_values("net_pnl")

show_table(tag_summary_main, "High Risk Tag Summary")


# ============================================================
# 8. Scenario proxy tests
# ============================================================
# Diagnostic only. Not final trading rules.
# ============================================================

scenario_notes = []
rd_q25 = df[rd_col].quantile(0.25) if rd_col else np.nan
tw50_q75 = df[tw50_col].quantile(0.75) if tw50_col else np.nan

def available(col):
    return col is not None and col in df.columns

scenarios = {}
scenarios["Base_v2B3"] = pd.Series(True, index=df.index)
scenarios["Exclude_LunchTail_diagnostic"] = ~df["is_Lunch_Tail_1245_1300"]
scenarios["FirstSignal_only_diagnostic"] = ~df["is_Repeated_Signal"]
scenarios["Exclude_HighRV_top20_diagnostic"] = ~df["is_High_RV10_Top20pct"]
scenarios["Exclude_HighRV_and_EXIT_TIME"] = ~(df["is_High_RV10_Top20pct"] & df["is_EXIT_TIME_No_Reversion"])
scenarios["Exclude_EXIT_TIME_diagnostic_not_tradable"] = ~df["is_EXIT_TIME_No_Reversion"]
scenarios["Strict_execution_proxy_not_tradable"] = ~df["is_Slip_Fragile_025tick"]

if available(rd_col):
    scenarios["LunchTail_keep_deeper_RD_only"] = (~df["is_Lunch_Tail_1245_1300"]) | (df["is_Lunch_Tail_1245_1300"] & (df[rd_col] <= rd_q25))
    scenarios["Repeated_require_deeper_RD"] = (~df["is_Repeated_Signal"]) | (df["is_Repeated_Signal"] & (df[rd_col] <= rd_q25))
else:
    scenario_notes.append("Feature missing: entry_relative_deviation. RD-based scenarios skipped.")

if available(tw50_col):
    scenarios["LunchTail_keep_stronger_TW50_only"] = (~df["is_Lunch_Tail_1245_1300"]) | (df["is_Lunch_Tail_1245_1300"] & (df[tw50_col] >= tw50_q75))
    scenarios["Repeated_require_stronger_TW50"] = (~df["is_Repeated_Signal"]) | (df["is_Repeated_Signal"] & (df[tw50_col] >= tw50_q75))
else:
    scenario_notes.append("Feature missing: tw50_return_5m. TW50-based scenarios skipped.")

if available(rd_col) and available(tw50_col):
    scenarios["LunchTail_keep_deepRD_or_strongT5"] = (~df["is_Lunch_Tail_1245_1300"]) | (df["is_Lunch_Tail_1245_1300"] & ((df[rd_col] <= rd_q25) | (df[tw50_col] >= tw50_q75)))
    scenarios["Repeated_require_deepRD_or_strongT5"] = (~df["is_Repeated_Signal"]) | (df["is_Repeated_Signal"] & ((df[rd_col] <= rd_q25) | (df[tw50_col] >= tw50_q75)))

scenario_rows = []
for name, mask in scenarios.items():
    sub = df[mask].copy()
    base = perf_summary(sub, "pnl_no_cost")
    c001 = perf_summary(sub, "pnl_cost_001")
    c002 = perf_summary(sub, "pnl_cost_002")
    s025 = perf_summary(sub, "pnl_slip_025tick")
    scenario_rows.append({
        "scenario": name,
        "trades": int(base["trades"].iloc[0]),
        "net_pnl_no_cost": base["net_pnl"].iloc[0],
        "avg_pnl_no_cost": base["avg_pnl"].iloc[0],
        "pf_no_cost": base["profit_factor"].iloc[0],
        "max_dd_no_cost": base["estimated_max_drawdown"].iloc[0],
        "net_pnl_cost001": c001["net_pnl"].iloc[0],
        "avg_pnl_cost001": c001["avg_pnl"].iloc[0],
        "pf_cost001": c001["profit_factor"].iloc[0],
        "net_pnl_cost002": c002["net_pnl"].iloc[0],
        "avg_pnl_cost002": c002["avg_pnl"].iloc[0],
        "pf_cost002": c002["profit_factor"].iloc[0],
        "max_dd_cost002": c002["estimated_max_drawdown"].iloc[0],
        "net_pnl_slip025": s025["net_pnl"].iloc[0],
        "pf_slip025": s025["profit_factor"].iloc[0],
    })

scenario_proxy = pd.DataFrame(scenario_rows).sort_values("net_pnl_cost002", ascending=False)
show_table(scenario_proxy, "Scenario Proxy Tests")

scenario_by_split_list = []
for name, mask in scenarios.items():
    sub = df[mask].copy()
    temp = perf_summary(sub, "pnl_cost_002", "data_split")
    temp.insert(0, "scenario", name)
    scenario_by_split_list.append(temp)
scenario_by_split = pd.concat(scenario_by_split_list, ignore_index=True).sort_values(["scenario", "data_split"])


# ============================================================
# 9. Casebooks
# ============================================================

case_cols = [
    "trade_id", "entry_dt", "exit_dt", "entry_date", "entry_time_hhmm",
    "entry_year", "entry_month", "data_split", "session_block", "exit_reason",
    "signal_count_today", "holding_minutes",
    "pnl_no_cost", "pnl_cost_001", "pnl_cost_002", "pnl_slip_025tick",
    "risk_tags",
]
for c in [rd_col, tw50_col, rv_col, slope_col]:
    if c and c not in case_cols:
        case_cols.append(c)
case_cols = [c for c in case_cols if c in df.columns]

worst_loss_base = df[case_cols].sort_values("pnl_no_cost", ascending=True).head(30)
worst_loss_cost002 = df[case_cols].sort_values("pnl_cost_002", ascending=True).head(30)
cost_fragile_cases = df[df["is_Cost_Fragile_001"] | df["is_Cost_Fragile_002"] | df["is_Slip_Fragile_025tick"]][case_cols].sort_values("pnl_no_cost", ascending=True)
exit_time_cases = df[df["is_EXIT_TIME_No_Reversion"]][case_cols].sort_values("pnl_cost_002", ascending=True)
repeated_cases = df[df["is_Repeated_Signal"]][case_cols].sort_values("pnl_cost_002", ascending=True)
lunch_cases = df[df["is_Lunch_Tail_1245_1300"]][case_cols].sort_values("pnl_cost_002", ascending=True)
highrv_cases = df[df["is_High_RV10_Top20pct"]][case_cols].sort_values("pnl_cost_002", ascending=True)


# ============================================================
# 10. File/data diagnostics
# ============================================================

diagnostics = pd.DataFrame({
    "item": [
        "input_file", "rows", "pnl_col", "rd_col", "tw50_col", "rv_col", "slope_col",
        "exit_reason_unique_count", "has_unknown_exit_only", "warning"
    ],
    "value": [
        input_file, len(df), pnl_col, rd_col, tw50_col, rv_col, slope_col,
        df["exit_reason"].nunique(dropna=False),
        bool(df["exit_reason"].astype(str).eq("Unknown_Raw_CSV").all()),
        "If rd_col/tw50_col/rv_col are missing, feature-based scenario tests are incomplete."
    ]
})

conclusion_text = f"""
0050 v2-B3 High Risk Sample Review - CLEAN FIXED Summary

1. This is not v2-C. It is a high-risk sample review for v2-B3 final signal candidate.
2. No-cost trades = {int(overall_no_cost.iloc[0]['trades'])}, Net PnL = {overall_no_cost.iloc[0]['net_pnl']:.4f}, PF = {overall_no_cost.iloc[0]['profit_factor']:.4f}.
3. After fixed cost 0.02, Net PnL = {overall_cost002.iloc[0]['net_pnl']:.4f}, PF = {overall_cost002.iloc[0]['profit_factor']:.4f}.
4. Lunch tail tagging has been fixed: only exact lunch-tail sessions are tagged.
5. Scenario proxy tests are diagnostic only. Do not choose a future rule only because PF is higher.
6. If feature columns such as entry_relative_deviation or tw50_return_5m are missing, deeper expected-edge tests must be rerun with a full-feature event dataset.
7. Use the output to write Main Finding, Known Limitations, and Future Work.
"""
conclusion_df = pd.DataFrame({"conclusion": [conclusion_text]})
notes_df = pd.DataFrame({"note": scenario_notes if scenario_notes else ["All feature-based scenario tests were available."]})
readme_df = pd.DataFrame({
    "section": ["Purpose", "Input", "Important Fix", "Warning", "Next Step"],
    "content": [
        "Review v2-B3 high-risk samples. This is not v2-C.",
        "Recommended input: 0050_v2B3_final_stratified_review_outputs.xlsx, sheet Parsed_B3_Events.",
        "Lunch tail tag now uses exact session matching and will not mis-tag 1000_1245 as lunch tail.",
        "Scenario_Proxy is diagnostic only, not final trading rule selection.",
        "Send Tag_Summary and Scenario_Proxy back for interpretation, or upload the output xlsx."
    ]
})


# ============================================================
# 11. Export
# ============================================================

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    readme_df.to_excel(writer, sheet_name="README", index=False)
    diagnostics.to_excel(writer, sheet_name="Diagnostics", index=False)
    notes_df.to_excel(writer, sheet_name="Scenario_Notes", index=False)

    df.to_excel(writer, sheet_name="Parsed_Events_With_Tags", index=False)

    overall_no_cost.to_excel(writer, sheet_name="Overall_NoCost", index=False)
    overall_cost001.to_excel(writer, sheet_name="Overall_Cost001", index=False)
    overall_cost002.to_excel(writer, sheet_name="Overall_Cost002", index=False)
    overall_slip025.to_excel(writer, sheet_name="Overall_Slip025", index=False)

    by_session.to_excel(writer, sheet_name="By_Session_NoCost", index=False)
    by_session_cost002.to_excel(writer, sheet_name="By_Session_Cost002", index=False)
    by_exit.to_excel(writer, sheet_name="By_Exit_NoCost", index=False)
    by_exit_cost002.to_excel(writer, sheet_name="By_Exit_Cost002", index=False)
    by_split.to_excel(writer, sheet_name="By_Split_NoCost", index=False)
    by_split_cost002.to_excel(writer, sheet_name="By_Split_Cost002", index=False)
    by_signal_count.to_excel(writer, sheet_name="By_Signal_NoCost", index=False)
    by_signal_count_cost002.to_excel(writer, sheet_name="By_Signal_Cost002", index=False)

    tag_summary_main.to_excel(writer, sheet_name="Tag_Summary", index=False)
    tag_x_split.to_excel(writer, sheet_name="Tag_x_Split_Cost002", index=False)
    tag_x_session.to_excel(writer, sheet_name="Tag_x_Session_Cost002", index=False)
    tag_x_exit.to_excel(writer, sheet_name="Tag_x_Exit_Cost002", index=False)

    scenario_proxy.to_excel(writer, sheet_name="Scenario_Proxy", index=False)
    scenario_by_split.to_excel(writer, sheet_name="Scenario_By_Split", index=False)

    worst_loss_base.to_excel(writer, sheet_name="Worst_Loss_Base", index=False)
    worst_loss_cost002.to_excel(writer, sheet_name="Worst_Loss_Cost002", index=False)
    cost_fragile_cases.to_excel(writer, sheet_name="Cost_Fragile_Cases", index=False)
    exit_time_cases.to_excel(writer, sheet_name="EXIT_TIME_Cases", index=False)
    repeated_cases.to_excel(writer, sheet_name="Repeated_Cases", index=False)
    lunch_cases.to_excel(writer, sheet_name="Lunch_Tail_Cases", index=False)
    highrv_cases.to_excel(writer, sheet_name="HighRV_Cases", index=False)

    conclusion_df.to_excel(writer, sheet_name="Conclusion", index=False)

print("\nDone. Exported:", OUTPUT_FILE)
if IS_COLAB:
    files.download(OUTPUT_FILE)


Please upload: 0050_v2B3_final_stratified_review_outputs.xlsx


Saving UPLOAD_THIS_TO_COLAB__v2B3_Parsed_B3_Events_223.csv to UPLOAD_THIS_TO_COLAB__v2B3_Parsed_B3_Events_223.csv
Loaded: UPLOAD_THIS_TO_COLAB__v2B3_Parsed_B3_Events_223.csv
Raw shape: (223, 48)
Raw columns: ['version', 'trade_number', 'entry_datetime', 'exit_datetime', 'entry_date', 'entry_year', 'entry_month', 'entry_time_hhmm', 'exit_time_hhmm', 'entry_price', 'exit_price', 'base_pnl_twd', 'pnl_cost_001', 'pnl_cost_002', 'pnl_cost_0035', 'holding_minutes', 'holding_bucket', 'exit_reason', 'mfe_twd', 'mae_twd', 'mode', 'entry_relative_deviation', 'volume_ratio_at_entry', 'deviation_slope_1bar', 'deviation_slope_3bar', 'etf_return_5m', 'etf_return_15m', 'tw50_return_5m', 'tw50_return_15m', 'realized_vol_10bar', 'minutes_since_open', 'minutes_to_close', 'session_id', 'signal_count_today', 't5_filter_ok', 'b1_filter_ok', 'b2_filter_ok', 'b3_filter_ok', 'exit_relative_deviation', 'bars_held', 'session_block', 'data_split', 'is_0945_1000_bucket', 'is_exact_0945', 'is_exact_1000', 'entry_s

/tmp/ipykernel_4407/3743808467.py:201: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return data.groupby(group_cols, dropna=False).apply(_calc).reset_index()
/tmp/ipykernel_4407/3743808467.py:201: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return data.groupby(group_cols, dropna=False).apply(_calc).reset_index()
/tmp/ipykernel_4407/3743808467.py:201: DeprecationWarning: DataFrameGroupBy.apply operated on 

,trades,wins,losses,flats,win_rate,loss_rate,net_pnl,avg_pnl,median_pnl,gross_profit,gross_loss,profit_factor,max_single_profit,max_single_loss,estimated_max_drawdown
0,223.0,92.0,39.0,92.0,0.412556,0.174888,7.8,0.034978,0.0,10.8,-3.0,3.6,0.65,-0.25,0.25



Overall - Cost 0.02


...

,scenario,trades,net_pnl_no_cost,avg_pnl_no_cost,pf_no_cost,max_dd_no_cost,net_pnl_cost001,avg_pnl_cost001,pf_cost001,net_pnl_cost002,avg_pnl_cost002,pf_cost002,max_dd_cost002,net_pnl_slip025,pf_slip025
9,LunchTail_keep_stronger_TW50_only,190,7.75,0.040789,4.444444,0.25,5.85,0.030789,2.767372,3.95,0.020789,1.903890,0.73,3.000,1.612245
11,LunchTail_keep_deepRD_or_strongT5,200,7.90,0.039500,4.361702,0.25,5.90,0.029500,2.695402,3.90,0.019500,1.845987,0.80,2.900,1.560386
8,Repeated_require_deeper_RD,179,7.45,0.041620,4.104167,0.25,5.66,0.031620,2.664706,3.87,0.021620,1.879545,0.58,2.975,1.607143
1,Exclude_LunchTail_diagnostic,183,7.50,0.040984,4.488372,0.30,5.67,0.030984,2.783019,3.84,0.020984,1.912114,0.93,2.925,1.619048
7,LunchTail_keep_deeper_RD_only,197,7.75,0.039340,4.297872,0.25,5.78,0.029340,2.665706,3.81,0.019340,1.830065,0.86,2.825,1.548544
12,Repeated_require_deepRD_or_strongT5,186,7.50,0.040323,3.941176,0.25,5.64,0.030323,2.566667,3.78,0.020323,1.812903,0.57,2.850,1.550725
10,Repeated_require_stronger_TW50,169,7.15,0.042308,3.979167,0.35,5.46,0.032308,2.629851,3.77,0.022308,1.876744,0.58,2.925,1.612565
2,FirstSignal_only_diagnostic,150,6.35,0.042333,4.097561,0.30,4.85,0.032333,2.678201,3.35,0.022333,1.898123,0.67,2.600,1.626506
0,Base_v2B3,223,7.80,0.034978,3.600000,0.25,5.57,0.024978,2.292343,3.34,0.014978,1.594306,0.95,2.225,1.354582
6,Strict_execution_proxy_not_tradable,223,7.80,0.034978,3.600000,0.25,5.57,0.024978,2.292343,3.34,0.014978,1.594306,0.95,2.225,1.354582


/tmp/ipykernel_4407/3743808467.py:201: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return data.groupby(group_cols, dropna=False).apply(_calc).reset_index()
/tmp/ipykernel_4407/3743808467.py:201: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return data.groupby(group_cols, dropna=False).apply(_calc).reset_index()
/tmp/ipykernel_4407/3743808467.py:201: DeprecationWarning: DataFrameGroupBy.apply operated on 


Done. Exported: 0050_v2B3_high_risk_sample_review_outputs_FIXED.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>